In [4]:
import glob
import numpy as np
import os
import sys

# add parent folder (production) to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.LoRa import MultiBAMv4

import importlib
import utils.my_lora_utils
importlib.reload(utils.my_lora_utils)
from utils.my_lora_utils import *

print(utils.my_lora_utils.__file__)

def parse_gt_from_filename(f):
    parts = f.split("_")
    return parts[6]

c:\Users\priba\Sean-2025\INC-LAB\BAM\INC-BAM\utils\my_lora_utils.py


In [ ]:
sf = 9       # Spreading Factor 
N = 2**sf
input_row = 512
input_col = 33

In [ ]:
dataset_name_folder = f"dataset_sf{sf}_{input_row}x{input_col}"
dataset_clean_name_folder = f"clean_dataset_sf{sf}_{input_row}x{input_col}"
input_layer = input_row * input_col
layers = [input_layer, 1024, 256]  # compress 3840 → 1024 → 256

GEENRATE_ = True #### Secure accidently running

################### Load all .npy files ########################################
print("LOAD DATASET")
files = glob.glob(f'{dataset_name_folder}/*.npy')
data_list = []
gt_list = []
database_clean_signal = []
for f in files:
    x = np.load(f)
    gt_symbol = parse_gt_from_filename(f)
    data_list.append(x.flatten())
    gt_list.append(int(gt_symbol)) 

for sym in range(N): # 0 until 2**sf
    file_str = f'{dataset_clean_name_folder}/s_sf{sf}_bw125_{sym}.npy'
    x = np.load(file_str)
    x_flat= x.flatten()
    database_clean_signal.append(x_flat)
    
X = np.array(data_list)
gt_list = np.array(gt_list)
database_clean_signal = np.array(database_clean_signal)

################### Load all .npy files ########################################

multi_bam = MultiBAMv4(layers_dims=layers, eta=1e-5)

if (GEENRATE_):
    
    folder_path = f"weight_{input_layer}_1024_256"

    # Check if folder exists, if not create it
    check_and_make_folder(folder_path)
    layer_losses = multi_bam.train(X=X, Y_sym = gt_list, database=database_clean_signal, num_epochs=10, batch_size=32)
    
    for i, bam in enumerate(multi_bam.bams):
        np.save(f"{folder_path}/weights_layer_{i}.npy", bam.W)

def load_weight(): 
    ## HOW TO LOAD WEIGHT
    layers = [input_row * input_col, 1024, 256] # <-- must match training

    multi_bam = MultiBAMv4(layers_dims=layers, eta=1e-5)
    for i, bam in enumerate(multi_bam.bams):
        bam.W = np.load(f"weight/weights_layer_{i}.npy")


LOAD DATASET
Folder already exists: weight_4224_1024_256

--- Training Layer 1/2 ---
tensor([1871, 5252, 2722, 5667, 2049, 5322, 2870, 2368, 4811, 3543, 5211, 5507,
        2748, 2448,  557, 3006, 3396, 3306, 3728,  521, 2894, 2896, 1379, 1893,
        2532, 4604, 2774,  756, 4415, 4111, 5091, 5458, 1373, 3476,  730, 2483,
        4863, 2775, 3478, 5013, 5552, 5478, 1086, 2985, 2282, 4184, 5259,  837,
        3013,  452, 2724,   46, 4363, 5248, 2002, 1203, 5471, 4775, 5594, 2879,
        2623,  674, 2062, 1877, 1665, 5729, 4737, 1804, 3970, 2711, 4911, 4836,
        1512, 4847, 2695, 4981, 1448, 3250, 1362, 1914,  821, 4472, 3569, 5394,
        3169, 1141,  530,  481, 2868, 1439,  555, 4057, 3722, 3508, 2444, 3419,
        2475, 2694,  685, 5195])
tensor([1871, 5252, 2722, 5667, 2049, 5322, 2870, 2368, 4811, 3543, 5211, 5507,
        2748, 2448,  557, 3006, 3396, 3306, 3728,  521, 2894, 2896, 1379, 1893,
        2532, 4604, 2774,  756, 4415, 4111, 5091, 5458, 1373, 3476,  730, 2483,
  